# 404 — Program Annotation

## Objective

Characterize the biological processes represented in the two transcriptomic consensus programs from Notebook 401. The analysis uses the complete signed gene-weight rankings across the shared universe of 2,388 genes. This allows gene sets to be associated with either end of each program's ranking.

## Annotation framework

The analysis uses three collections from MSigDB v2026.1.Hs. Hallmark provides the primary annotation, Reactome adds pathway detail, and GO Biological Process provides exploratory context. Gene sets are eligible when 10 to 500 of their genes are present in the shared universe.

The notebook reports the direction and magnitude of each association and exports all eligible tests with their adjusted p-values. These annotations describe the biological themes represented in the signed consensus rankings.

In [1]:
# =============================================================================
# Imports
# =============================================================================

import numpy as np
import pandas as pd

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

from pancancer_epigenetics.analysis_parameters import load_consensus_programs
from pancancer_epigenetics.utils.paths import Paths

In [2]:
# =============================================================================
# Input and output directories
# =============================================================================

CONSENSUS_PROGRAM_DIR = Paths.consensus_programs
OUTPUT_DIR = Paths.consensus_programs
MSIGDB_DIR = Paths.msigdb

In [3]:
# =============================================================================
# Authoritative program-annotation input paths
# =============================================================================

CONSENSUS_GENE_WEIGHTS_PATH = (
    CONSENSUS_PROGRAM_DIR / "401_consensus_transcriptomic_gene_weights.csv"
)

In [4]:
# =============================================================================
# Load authoritative program-annotation inputs
# =============================================================================

consensus_program_ids = tuple(
    program.program_id for program in load_consensus_programs()
)

consensus_gene_weights = pd.read_csv(CONSENSUS_GENE_WEIGHTS_PATH)

weight_program_ids = set(consensus_gene_weights["consensus_program_id"])

if set(consensus_program_ids) != weight_program_ids:
    raise ValueError(
        "The Notebook 401 catalog and gene weights list different programs."
    )

genes_per_program = consensus_gene_weights.groupby("consensus_program_id")[
    "gene_symbol"
].nunique()

print(genes_per_program.reindex(list(consensus_program_ids)))

consensus_program_id
CONSENSUS_TX_01    2388
CONSENSUS_TX_02    2388
Name: gene_symbol, dtype: int64


In [5]:
# =============================================================================
# Define catalog ordering for annotation tables
# =============================================================================


def program_sort_key(column):
    if column.name == "consensus_program_id":
        return pd.Categorical(column, categories=consensus_program_ids, ordered=True)
    return column

,consensus_program_id,tumor_rna_axis,cell_line_program,orientation_multiplier
0,CONSENSUS_TX_01,RNA_IC150,ICA_PROGRAM_15,1
1,CONSENSUS_TX_02,RNA_IC151,ICA_PROGRAM_47,-1


In [6]:
# =============================================================================
# Freeze signed consensus-program ranking
# =============================================================================

ranked_consensus_gene_weights = (
    consensus_gene_weights.loc[
        consensus_gene_weights["consensus_program_id"].isin(consensus_program_ids),
        [
            "consensus_program_id",
            "gene_symbol",
            "consensus_weight",
        ],
    ]
    .sort_values(
        ["consensus_program_id", "consensus_weight"],
        ascending=[True, False],
        key=program_sort_key,
    )
    .reset_index(drop=True)
)

In [7]:
# =============================================================================
# Freeze consensus-gene annotation universe
# =============================================================================

consensus_gene_universe = (
    ranked_consensus_gene_weights[["gene_symbol"]]
    .drop_duplicates()
    .sort_values("gene_symbol")
    .reset_index(drop=True)
)

consensus_gene_universe.shape

(2388, 1)

In [8]:
# =============================================================================
# Freeze biological-annotation design
# =============================================================================

MSIGDB_RELEASE = "v2026.1.Hs"

ANNOTATION_COLLECTIONS = {
    "HALLMARK": {
        "filename": "h.all.v2026.1.Hs.symbols.gmt",
        "analysis_tier": "primary",
    },
    "REACTOME": {
        "filename": "c2.cp.reactome.v2026.1.Hs.symbols.gmt",
        "analysis_tier": "secondary",
    },
    "GO_BP": {
        "filename": "c5.go.bp.v2026.1.Hs.symbols.gmt",
        "analysis_tier": "exploratory",
    },
}

MIN_GENE_SET_SIZE = 10
MAX_GENE_SET_SIZE = 500
FDR_ALPHA = 0.05

In [9]:
# =============================================================================
# MSigDB annotation resource paths
# =============================================================================

annotation_resource_paths = {
    collection_name: MSIGDB_DIR / collection_config["filename"]
    for collection_name, collection_config in ANNOTATION_COLLECTIONS.items()
}

In [10]:
# =============================================================================
# Define GMT parser
# =============================================================================

def read_gmt(path):
    """Read an MSigDB GMT file as a mapping from gene-set name to symbols."""
    gene_sets = {}

    with path.open(encoding="utf-8") as handle:
        for line in handle:
            fields = line.rstrip("\n").split("\t")
            gene_sets[fields[0]] = set(fields[2:])

    return gene_sets

In [11]:
# =============================================================================
# Load MSigDB annotation collections
# =============================================================================

annotation_gene_sets = {
    collection_name: read_gmt(resource_path)
    for collection_name, resource_path in annotation_resource_paths.items()
}

In [12]:
# =============================================================================
# Summarize loaded annotation collections
# =============================================================================

annotation_collection_summary = pd.DataFrame(
    [
        {
            "collection": collection_name,
            "analysis_tier": ANNOTATION_COLLECTIONS[collection_name]["analysis_tier"],
            "gene_set_count": len(gene_sets),
        }
        for collection_name, gene_sets in annotation_gene_sets.items()
    ]
)

annotation_collection_summary

,collection,analysis_tier,gene_set_count
0,HALLMARK,primary,50
1,REACTOME,secondary,1839
2,GO_BP,exploratory,7538


In [13]:
# =============================================================================
# Restrict gene sets to the frozen annotation universe
# =============================================================================

consensus_gene_universe_set = set(consensus_gene_universe["gene_symbol"])

eligible_annotation_gene_sets = {}

for collection_name, gene_sets in annotation_gene_sets.items():
    eligible_annotation_gene_sets[collection_name] = {}

    for gene_set_name, genes in gene_sets.items():
        overlapping_genes = genes & consensus_gene_universe_set

        if MIN_GENE_SET_SIZE <= len(overlapping_genes) <= MAX_GENE_SET_SIZE:
            eligible_annotation_gene_sets[collection_name][gene_set_name] = (
                overlapping_genes
            )

In [14]:
# =============================================================================
# Summarize eligible annotation gene sets
# =============================================================================

eligible_annotation_summary = pd.DataFrame(
    [
        {
            "collection": collection_name,
            "analysis_tier": ANNOTATION_COLLECTIONS[collection_name]["analysis_tier"],
            "eligible_gene_set_count": len(gene_sets),
        }
        for collection_name, gene_sets in eligible_annotation_gene_sets.items()
    ]
)

eligible_annotation_summary

,collection,analysis_tier,eligible_gene_set_count
0,HALLMARK,primary,37
1,REACTOME,secondary,272
2,GO_BP,exploratory,2062


## Frozen inferential design

Program annotation uses a competitive rank-based test on the complete signed
consensus-gene profile.

For each consensus program and eligible gene set:

- genes inside the set are compared with all remaining genes in the shared gene universe defined above;
- the test is two-sided Mann–Whitney U, allowing enrichment toward either the
  positive or negative end of the signed consensus ranking;
- effect size is reported as signed rank-biserial correlation;
- no top-gene threshold is introduced;
- Benjamini–Hochberg correction is applied separately within each
  consensus-program × annotation-collection family;
- Hallmark is the primary annotation layer, Reactome is secondary, and GO
  Biological Process is exploratory.

Statistical enrichment is interpreted as preferential representation within the
frozen transcriptomic program, not as pathway activation, causal regulation, or
biological validation.

In [15]:
# =============================================================================
# Define competitive rank-based enrichment test
# =============================================================================

def test_rank_enrichment(program_weights, gene_set):
    """Test whether a gene set is displaced within a signed program ranking."""
    in_set = program_weights.loc[
        program_weights["gene_symbol"].isin(gene_set),
        "consensus_weight",
    ]
    out_set = program_weights.loc[
        ~program_weights["gene_symbol"].isin(gene_set),
        "consensus_weight",
    ]

    test = mannwhitneyu(
        in_set,
        out_set,
        alternative="two-sided",
    )

    rank_biserial = (
        2 * test.statistic / (len(in_set) * len(out_set))
        - 1
    )

    return test.statistic, test.pvalue, rank_biserial

In [16]:
# =============================================================================
# Compute raw rank-based annotation results
# =============================================================================

annotation_results = []

for program_id, program_weights in ranked_consensus_gene_weights.groupby(
    "consensus_program_id",
    sort=False,
):
    for collection_name, gene_sets in eligible_annotation_gene_sets.items():
        for gene_set_name, gene_set in gene_sets.items():
            statistic, p_value, rank_biserial = test_rank_enrichment(
                program_weights,
                gene_set,
            )

            annotation_results.append(
                {
                    "consensus_program_id": program_id,
                    "collection": collection_name,
                    "analysis_tier": ANNOTATION_COLLECTIONS[
                        collection_name
                    ]["analysis_tier"],
                    "gene_set": gene_set_name,
                    "gene_set_size": len(gene_set),
                    "mann_whitney_u": statistic,
                    "p_value": p_value,
                    "rank_biserial": rank_biserial,
                }
            )

annotation_results = pd.DataFrame(annotation_results)

In [17]:
# =============================================================================
# Apply within-family multiple-testing correction
# =============================================================================

annotation_results["q_value"] = np.nan

for _, family_index in annotation_results.groupby(
    ["consensus_program_id", "collection"]
).groups.items():
    annotation_results.loc[family_index, "q_value"] = multipletests(
        annotation_results.loc[family_index, "p_value"],
        method="fdr_bh",
    )[1]

In [18]:
# =============================================================================
# Summarize annotation results by program and collection
# =============================================================================

annotation_result_summary = (
    annotation_results.assign(q_lt_0_05=annotation_results["q_value"].lt(FDR_ALPHA))
    .groupby(
        ["consensus_program_id", "collection", "analysis_tier"],
        as_index=False,
    )
    .agg(
        tested_gene_sets=("gene_set", "size"),
        q_lt_0_05=("q_lt_0_05", "sum"),
        min_rank_biserial=("rank_biserial", "min"),
        max_rank_biserial=("rank_biserial", "max"),
    )
    .sort_values(
        ["consensus_program_id", "collection", "analysis_tier"],
        key=program_sort_key,
        ignore_index=True,
    )
)

annotation_result_summary

,consensus_program_id,collection,analysis_tier,tested_gene_sets,q_lt_0_05,min_rank_biserial,max_rank_biserial
0,CONSENSUS_TX_01,GO_BP,exploratory,2062,462,-0.612372,0.951581
1,CONSENSUS_TX_01,HALLMARK,primary,37,18,-0.506779,0.625146
2,CONSENSUS_TX_01,REACTOME,secondary,272,77,-0.828202,0.937899
3,CONSENSUS_TX_02,GO_BP,exploratory,2062,34,-0.741892,0.532969
4,CONSENSUS_TX_02,HALLMARK,primary,37,9,-0.661416,0.448860
5,CONSENSUS_TX_02,REACTOME,secondary,272,19,-0.739803,0.375870


In [19]:
# =============================================================================
# Inspect supported primary Hallmark annotations
# =============================================================================

hallmark_supported_terms = (
    annotation_results.loc[
        (annotation_results["collection"] == "HALLMARK")
        & annotation_results["q_value"].lt(FDR_ALPHA)
    ]
    .assign(
        direction=lambda df: np.where(
            df["rank_biserial"] > 0,
            "positive",
            "negative",
        ),
        abs_rank_biserial=lambda df: df["rank_biserial"].abs(),
    )
    .sort_values(
        ["consensus_program_id", "abs_rank_biserial"],
        ascending=[True, False],
        key=program_sort_key,
    )[
        [
            "consensus_program_id",
            "gene_set",
            "gene_set_size",
            "direction",
            "rank_biserial",
            "p_value",
            "q_value",
        ]
    ]
)

hallmark_supported_terms

,consensus_program_id,gene_set,gene_set_size,direction,rank_biserial,p_value,q_value
1,CONSENSUS_TX_01,HALLMARK_ALLOGRAFT_REJECTION,53,positive,0.625146,6.538630e-15,1.209646e-13
34,CONSENSUS_TX_01,HALLMARK_UV_RESPONSE_DN,36,negative,-0.506779,1.736492e-07,1.606255e-06
11,CONSENSUS_TX_01,HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION,114,negative,-0.497716,2.706433e-19,1.001380e-17
23,CONSENSUS_TX_01,HALLMARK_INTERFERON_GAMMA_RESPONSE,50,positive,0.466262,1.612610e-08,1.988886e-07
15,CONSENSUS_TX_01,HALLMARK_GLYCOLYSIS,39,negative,-0.384637,3.701020e-05,1.711722e-04
0,CONSENSUS_TX_01,HALLMARK_ADIPOGENESIS,20,negative,-0.377280,3.621627e-03,1.030771e-02
22,CONSENSUS_TX_01,HALLMARK_INTERFERON_ALPHA_RESPONSE,19,positive,0.365000,6.070277e-03,1.321178e-02
2,CONSENSUS_TX_01,HALLMARK_ANDROGEN_RESPONSE,19,negative,-0.357268,7.234889e-03,1.487172e-02
28,CONSENSUS_TX_01,HALLMARK_P53_PATHWAY,35,negative,-0.349791,3.750363e-04,1.541816e-03
18,CONSENSUS_TX_01,HALLMARK_HYPOXIA,58,negative,-0.333698,1.380163e-05,7.295149e-05


In [20]:
# =============================================================================
# Inspect strongest supported secondary Reactome annotations
# =============================================================================

reactome_supported_terms = (
    annotation_results.loc[
        (annotation_results["collection"] == "REACTOME")
        & annotation_results["q_value"].lt(FDR_ALPHA)
    ]
    .assign(
        direction=lambda df: np.where(
            df["rank_biserial"] > 0,
            "positive",
            "negative",
        ),
        abs_rank_biserial=lambda df: df["rank_biserial"].abs(),
    )
    .sort_values(
        ["consensus_program_id", "abs_rank_biserial"],
        ascending=[True, False],
        key=program_sort_key,
    )
)

(
    reactome_supported_terms.groupby("consensus_program_id", group_keys=False).head(15)[
        [
            "consensus_program_id",
            "gene_set",
            "gene_set_size",
            "direction",
            "rank_biserial",
            "q_value",
        ]
    ]
)

,consensus_program_id,gene_set,gene_set_size,direction,rank_biserial,q_value
221,CONSENSUS_TX_01,REACTOME_PHOSPHORYLATION_OF_CD3_AND_TCR_ZETA_C...,14,positive,0.937899,7.353824e-08
139,CONSENSUS_TX_01,REACTOME_GENERATION_OF_SECOND_MESSENGER_MOLECULES,14,positive,0.924239,7.889033e-08
114,CONSENSUS_TX_01,REACTOME_DOWNSTREAM_TCR_SIGNALING,14,positive,0.921711,7.889033e-08
287,CONSENSUS_TX_01,REACTOME_TCR_SIGNALING,18,positive,0.918331,2.450060e-09
54,CONSENSUS_TX_01,REACTOME_ATTACHMENT_OF_BACTERIA_TO_EPITHELIAL_...,11,negative,-0.828202,2.694314e-05
174,CONSENSUS_TX_01,REACTOME_LAMININ_INTERACTIONS,12,negative,-0.812921,1.738103e-05
285,CONSENSUS_TX_01,REACTOME_SYNDECAN_INTERACTIONS,14,negative,-0.721567,3.862810e-05
187,CONSENSUS_TX_01,REACTOME_MET_ACTIVATES_PTK2_SIGNALING,17,negative,-0.672166,2.485487e-05
188,CONSENSUS_TX_01,REACTOME_MET_PROMOTES_CELL_MOTILITY,18,negative,-0.671730,1.498958e-05
205,CONSENSUS_TX_01,REACTOME_NON_INTEGRIN_MEMBRANE_ECM_INTERACTIONS,32,negative,-0.637442,5.047881e-08


In [21]:
# =============================================================================
# Inspect strongest supported exploratory GO-BP annotations
# =============================================================================

go_bp_supported_terms = (
    annotation_results.loc[
        (annotation_results["collection"] == "GO_BP")
        & annotation_results["q_value"].lt(FDR_ALPHA)
    ]
    .assign(
        direction=lambda df: np.where(
            df["rank_biserial"] > 0,
            "positive",
            "negative",
        ),
        abs_rank_biserial=lambda df: df["rank_biserial"].abs(),
    )
    .sort_values(
        ["consensus_program_id", "abs_rank_biserial"],
        ascending=[True, False],
        key=program_sort_key,
    )
)

(
    go_bp_supported_terms.groupby("consensus_program_id", group_keys=False).head(15)[
        [
            "consensus_program_id",
            "gene_set",
            "gene_set_size",
            "direction",
            "rank_biserial",
            "q_value",
        ]
    ]
)

,consensus_program_id,gene_set,gene_set_size,direction,rank_biserial,q_value
1377,CONSENSUS_TX_01,GOBP_PEPTIDE_ANTIGEN_ASSEMBLY_WITH_MHC_CLASS_I...,11,positive,0.951581,1.264680e-06
1378,CONSENSUS_TX_01,GOBP_PEPTIDE_ANTIGEN_ASSEMBLY_WITH_MHC_PROTEIN...,11,positive,0.951581,1.264680e-06
436,CONSENSUS_TX_01,GOBP_B_CELL_RECEPTOR_SIGNALING_PATHWAY,19,positive,0.890960,1.208429e-09
1725,CONSENSUS_TX_01,GOBP_REGULATION_OF_ANTIGEN_RECEPTOR_MEDIATED_S...,19,positive,0.822221,2.976975e-08
369,CONSENSUS_TX_01,GOBP_ANTIGEN_PROCESSING_AND_PRESENTATION_OF_EX...,16,positive,0.801855,8.207392e-07
370,CONSENSUS_TX_01,GOBP_ANTIGEN_PROCESSING_AND_PRESENTATION_OF_EX...,16,positive,0.801855,8.207392e-07
371,CONSENSUS_TX_01,GOBP_ANTIGEN_PROCESSING_AND_PRESENTATION_OF_EX...,16,positive,0.801855,8.207392e-07
374,CONSENSUS_TX_01,GOBP_ANTIGEN_RECEPTOR_MEDIATED_SIGNALING_PATHWAY,45,positive,0.800503,6.667846e-18
435,CONSENSUS_TX_01,GOBP_B_CELL_PROLIFERATION,26,positive,0.794600,1.873419e-10
2060,CONSENSUS_TX_01,GOBP_REGULATION_OF_T_CELL_RECEPTOR_SIGNALING_P...,14,positive,0.774461,1.174995e-05


In [22]:
# =============================================================================
# Consolidate supported biological annotations
# =============================================================================

supported_annotation_results = (
    annotation_results.loc[annotation_results["q_value"].lt(FDR_ALPHA)]
    .assign(
        direction=lambda df: np.where(
            df["rank_biserial"] > 0,
            "positive",
            "negative",
        ),
        abs_rank_biserial=lambda df: df["rank_biserial"].abs(),
    )
    .sort_values(
        [
            "consensus_program_id",
            "analysis_tier",
            "abs_rank_biserial",
        ],
        ascending=[True, True, False],
    )
    .reset_index(drop=True)
)  # Keep tied effects in the established order for the compact summary.
supported_annotation_results.sort_values(
    "consensus_program_id",
    key=program_sort_key,
    kind="stable",
    ignore_index=True,
)

In [23]:
# =============================================================================
# Build compact program-level annotation summary
# =============================================================================

supported_annotation_counts = (
    supported_annotation_results.groupby(
        [
            "consensus_program_id",
            "collection",
            "analysis_tier",
            "direction",
        ],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "supported_term_count"})
)

program_annotation_summary = (
    supported_annotation_results.sort_values("abs_rank_biserial", ascending=False)
    .groupby(
        [
            "consensus_program_id",
            "collection",
            "analysis_tier",
            "direction",
        ],
        as_index=False,
    )
    .first()
    .merge(
        supported_annotation_counts,
        on=[
            "consensus_program_id",
            "collection",
            "analysis_tier",
            "direction",
        ],
        how="left",
    )[
        [
            "consensus_program_id",
            "collection",
            "analysis_tier",
            "direction",
            "supported_term_count",
            "gene_set",
            "gene_set_size",
            "rank_biserial",
            "q_value",
        ]
    ]
    .sort_values(
        ["consensus_program_id", "collection", "analysis_tier", "direction"],
        key=program_sort_key,
        ignore_index=True,
    )
)

program_annotation_summary

,consensus_program_id,collection,analysis_tier,direction,supported_term_count,gene_set,gene_set_size,rank_biserial,q_value
0,CONSENSUS_TX_01,GO_BP,exploratory,negative,270,GOBP_CELLULAR_RESPONSE_TO_FATTY_ACID,13,-0.612372,1.409324e-03
1,CONSENSUS_TX_01,GO_BP,exploratory,positive,192,GOBP_PEPTIDE_ANTIGEN_ASSEMBLY_WITH_MHC_CLASS_I...,11,0.951581,1.264680e-06
2,CONSENSUS_TX_01,HALLMARK,primary,negative,13,HALLMARK_UV_RESPONSE_DN,36,-0.506779,1.606255e-06
3,CONSENSUS_TX_01,HALLMARK,primary,positive,5,HALLMARK_ALLOGRAFT_REJECTION,53,0.625146,1.209646e-13
4,CONSENSUS_TX_01,REACTOME,secondary,negative,59,REACTOME_ATTACHMENT_OF_BACTERIA_TO_EPITHELIAL_...,11,-0.828202,2.694314e-05
5,CONSENSUS_TX_01,REACTOME,secondary,positive,18,REACTOME_PHOSPHORYLATION_OF_CD3_AND_TCR_ZETA_C...,14,0.937899,7.353824e-08
6,CONSENSUS_TX_02,GO_BP,exploratory,negative,30,GOBP_KERATINIZATION,24,-0.741892,1.567503e-07
7,CONSENSUS_TX_02,GO_BP,exploratory,positive,4,GOBP_POSTSYNAPSE_ORGANIZATION,51,0.290812,2.660316e-02
8,CONSENSUS_TX_02,HALLMARK,primary,negative,6,HALLMARK_INTERFERON_ALPHA_RESPONSE,19,-0.661416,1.622476e-05
9,CONSENSUS_TX_02,HALLMARK,primary,positive,3,HALLMARK_SPERMATOGENESIS,20,0.448860,4.846693e-03


In [24]:
# =============================================================================
# Prepare authoritative program-annotation output
# =============================================================================

program_annotation_output = (
    annotation_results.assign(
        direction=lambda df: np.where(
            df["rank_biserial"] > 0,
            "positive",
            "negative",
        )
    )[
        [
            "consensus_program_id",
            "collection",
            "analysis_tier",
            "gene_set",
            "gene_set_size",
            "direction",
            "rank_biserial",
            "mann_whitney_u",
            "p_value",
            "q_value",
        ]
    ]
    .sort_values(
        [
            "consensus_program_id",
            "collection",
            "q_value",
            "gene_set",
        ],
        key=program_sort_key,
    )
    .reset_index(drop=True)
)

program_annotation_output.shape

(4742, 10)

In [25]:
# =============================================================================
# Program-annotation output path
# =============================================================================

PROGRAM_ANNOTATION_OUTPUT_PATH = (
    OUTPUT_DIR
    / "404_program_annotation_enrichment.csv"
)

In [26]:
# =============================================================================
# Write program-annotation artifact
# =============================================================================

program_annotation_output.to_csv(
    PROGRAM_ANNOTATION_OUTPUT_PATH,
    index=False,
)

In [27]:
# =============================================================================
# Verify written program-annotation artifact
# =============================================================================

reloaded_program_annotation_output = pd.read_csv(
    PROGRAM_ANNOTATION_OUTPUT_PATH
)

reloaded_program_annotation_output.shape

(4742, 10)

## Program annotation summary

At BH q < 0.05, the two consensus programs showed the following numbers of supported terms. Correction was applied separately for each program and collection.

| Consensus program | Hallmark (37 tested) | Reactome (272 tested) | GO Biological Process (2,062 tested) |
|---|---:|---:|---:|
| `CONSENSUS_TX_01` | 18 | 77 | 462 |
| `CONSENSUS_TX_02` | 9 | 19 | 34 |

**`CONSENSUS_TX_01`.** The positive end of the ranking showed a consistent immune-associated pattern, including antigen presentation, T-cell receptor signaling, and interferon responses. Epithelial–mesenchymal transition and extracellular-matrix interactions were associated with the negative end. The immune annotation should be interpreted alongside the strong influence of Lymphoid cell lines on global Pearson fidelity in Notebook 402.

**`CONSENSUS_TX_02`.** Keratinization, epidermal differentiation, and interferon-related terms were concentrated toward the negative end. Epithelial–mesenchymal transition was associated with the positive end in Hallmark, where three positive terms met the FDR threshold.

`404_program_annotation_enrichment.csv` contains all 4,742 eligible program–gene-set tests, with signed effect sizes and adjusted p-values.

In [ ]:
# =============================================================================
# Verify and register artifacts
# =============================================================================

from pancancer_epigenetics.utils.artifact_registry import (
    load_artifact_registry,
    refresh_notebook_artifacts,
)

output_objects = {
    PROGRAM_ANNOTATION_OUTPUT_PATH: program_annotation_output,
}
verified_outputs = {}
for path, expected in output_objects.items():
    if not path.is_file():
        raise FileNotFoundError(path)
    if path.suffix == ".json":
        with path.open(encoding="utf-8") as handle:
            observed = json.load(handle)
        if observed != expected:
            raise ValueError(f"Written metadata differs: {path}")
        verified_outputs[path] = None
    else:
        observed = (
            pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
        )
        if observed.shape != expected.shape or not observed.columns.equals(
            expected.columns
        ):
            raise ValueError(f"Written table structure differs: {path}")
        verified_outputs[path] = observed.shape

registered_count = refresh_notebook_artifacts(
    "notebooks/phase4_consensus_programs/404_program_annotation.ipynb",
    verified_outputs,
    require_unchanged=False,
)
load_artifact_registry()
print(f"Registered {registered_count} verified artifacts.")
